# ⚡ CircuitAI — Colab GPU Backend
**Step 1**: Runtime → Change runtime type → T4 or L4 GPU

**Step 2**: Run cells in order.

**Step 3**: Wait for `✅ All models ready`, then paste the `trycloudflare.com` URL into the frontend.

In [ ]:
# ── Cell 0: Mount Drive & Set Working Directory ─────────────────
import sys, os
from google.colab import drive

drive.mount('/content/drive')

# ⚠️ Update this path if your folder is in a different location
BACKEND_DIR = '/content/drive/MyDrive/knowledge_graph/backend'

os.chdir(BACKEND_DIR)
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

print(f'Working directory: {os.getcwd()}')
print(f'Files: {os.listdir()}')

In [ ]:
# ── Cell 1: Install Cloudflared ────────────────────────────────
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -o cloudflared.deb
!dpkg -i cloudflared.deb
print('Cloudflared installed ✓')

In [ ]:
# ── Cell 2: Install Python dependencies ────────────────────────
!pip install -r requirements.txt -q
print('Python deps installed ✓')

In [ ]:
# ── Cell 3: Start FastAPI server + Cloudflare Tunnel ───────────
import subprocess, threading, uvicorn
from server_utils import resolve_port, acquire_lock, wait_for_server
from main import app

# Step 1: Resolve port (kill old process if needed, fallback if still busy)
port = resolve_port(preferred=8000)

# Step 2: Acquire lock — prevents duplicate instances
acquire_lock()

# Step 3: Start uvicorn in a daemon thread
def start_server():
    uvicorn.run(app, host='0.0.0.0', port=port, log_level='info')

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
print(f'[startup] Server thread started on port {port}')

# Step 4: Wait for /health to confirm server + models are ready
ok = wait_for_server(port, timeout=120)
if not ok:
    raise RuntimeError('Server did not start in time. Check logs above.')

# Step 5: Start Cloudflare tunnel (only AFTER server is healthy)
print('\nStarting Cloudflare tunnel...')
p = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', f'http://localhost:{port}'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in p.stdout:
    print(line, end='')
    if 'trycloudflare.com' in line:
        tokens = line.split()
        public_url = next((t for t in tokens if 'trycloudflare.com' in t), None)
        if public_url:
            print('\n' + '='*60)
            print(f'  🔥 PUBLIC URL: {public_url}')
            print(f'  Backend port : {port}')
            print('  Paste this URL into CircuitAI frontend > Connection field')
            print('='*60)
        break